# 02 — Eval & Compare

Two-stage eval to save units:

* **Quick eval** — 1 episode/task × 10 tasks per suite × 4 suites = 40 episodes. ~5–10 min on L4. Noisy but cheap. Run this on every checkpoint.
* **Full eval** — 10 episodes/task × 10 tasks × 4 suites = 400 episodes. ~2–4h on L4. Run this only on configs that pass the quick check.

Held-out flow-matching loss is *not* a winner-selection signal (your own libero_sim_summary.md proved 0.3% loss improvement / 100→0% sim regression). Sim is the only truth here.

## 1. Pick a checkpoint to eval

In [ ]:
# === EDIT THIS ===
CONFIG = 'configs/libero_v5_run0_diagnostic.yaml'
MODE = 'quick'   # 'quick' (40 eps total) or 'full' (400 eps total)
# =================

import os, glob, yaml, json
REPO_DIR = '/content/memory-smolVLA'
PROJECT_ROOT = '/content/drive/MyDrive/memory-smolvla'
%cd {REPO_DIR}
os.environ['MUJOCO_GL'] = 'osmesa'

with open(CONFIG) as f:
    cfg = yaml.safe_load(f)
if '_base_' in cfg:
    base_path = os.path.join(os.path.dirname(CONFIG), cfg.pop('_base_'))
    with open(base_path) as f:
        base = yaml.safe_load(f)
    def merge(a, b):
        out = dict(a)
        for k, v in b.items():
            out[k] = merge(out.get(k, {}), v) if isinstance(v, dict) and isinstance(out.get(k), dict) else v
        return out
    cfg = merge(base, cfg)

ckpt_dir = cfg['trainer']['checkpoint_dir']
run_name = cfg['trainer'].get('wandb_run_name', os.path.basename(CONFIG).replace('.yaml', ''))

final_pt = os.path.join(ckpt_dir, 'final.pt')
if os.path.exists(final_pt):
    CKPT = final_pt
else:
    pts = sorted(glob.glob(os.path.join(ckpt_dir, 'step_*.pt')))
    if not pts:
        raise FileNotFoundError(f'No checkpoints in {ckpt_dir}. Run 01_train.ipynb first.')
    CKPT = pts[-1]

print(f'Evaluating: {CKPT}')
print(f'Mode:       {MODE}')
print(f'Run name:   {run_name}')

if MODE == 'quick':
    N_ROLLOUTS = 1
elif MODE == 'full':
    N_ROLLOUTS = 10
else:
    raise ValueError(MODE)

## 2. Run eval on all four LIBERO suites

scripts/eval.py writes per-suite JSON to `--output-dir`. We aggregate after.

In [ ]:
OUT_DIR = f'{PROJECT_ROOT}/results/{run_name}_{MODE}'
os.makedirs(OUT_DIR, exist_ok=True)

for suite in ['libero_spatial', 'libero_object', 'libero_goal', 'libero_10']:
    print(f'\n=== {suite} ===')
    !python scripts/eval.py \
        --checkpoint {CKPT} \
        --config {CONFIG} \
        --suite {suite} \
        --n-rollouts {N_ROLLOUTS} \
        --output-dir {OUT_DIR}

## 3. Aggregate results

Reads each suite's JSON, computes per-suite success and overall, prints a comparison row vs baseline v2.

In [ ]:
import json, glob
BASELINE_V2 = {'libero_spatial': 84.0, 'libero_object': 99.0, 'libero_goal': 96.0, 'libero_10': 72.0, 'overall': 87.75}
V4_BYPASS  = {'libero_spatial': 72.0, 'libero_object': 96.0, 'libero_goal': 82.0, 'libero_10': 54.0, 'overall': 76.00}
V4_MEMORY  = {'libero_spatial': 74.0, 'libero_object': 96.0, 'libero_goal': 79.0, 'libero_10': 44.0, 'overall': 73.25}

results = {}
for suite in ['libero_spatial', 'libero_object', 'libero_goal', 'libero_10']:
    paths = glob.glob(f'{OUT_DIR}/{suite}*.json') + glob.glob(f'{OUT_DIR}/*{suite}*.json')
    if not paths:
        print(f'  {suite}: no JSON found in {OUT_DIR}')
        continue
    with open(paths[0]) as f:
        data = json.load(f)
    per_task = data.get('per_task', {})
    if per_task:
        rates = [v['success_rate'] for v in per_task.values()]
        results[suite] = 100.0 * sum(rates) / len(rates)
    else:
        results[suite] = 0.0

if results:
    overall = sum(results.values()) / len(results)
    results['overall'] = overall
    
    print(f'\n{"Suite":<18} {"This run":>10} {"Baseline":>10} {"v4-bypass":>10} {"v4-memory":>10}  {"Δ vs base":>10}')
    print('-' * 72)
    for suite in ['libero_spatial', 'libero_object', 'libero_goal', 'libero_10', 'overall']:
        ours = results.get(suite, 0)
        b = BASELINE_V2[suite]
        bp = V4_BYPASS[suite]
        bm = V4_MEMORY[suite]
        delta = ours - b
        print(f'{suite:<18} {ours:>10.1f} {b:>10.1f} {bp:>10.1f} {bm:>10.1f}  {delta:>+10.1f}')
    
    summary_path = f'{OUT_DIR}/summary.json'
    with open(summary_path, 'w') as f:
        json.dump({'run_name': run_name, 'mode': MODE, 'checkpoint': CKPT,
                  'per_suite': results, 'baseline_v2': BASELINE_V2,
                  'v4_bypass': V4_BYPASS, 'v4_memory': V4_MEMORY}, f, indent=2)
    print(f'\nSummary saved: {summary_path}')

## 4. Decision rules

**Run 0 (diagnostic):**
* Quick-eval **overall ≥ 80** → batch fix alone closed most of the gap. Run 1 may help further; if it doesn't, Run 0 is the cheap winner.
* Quick-eval **overall < 80** → batch fix alone insufficient. Move to Run 1.

**Run 1 (kitchen sink):**
* Quick-eval **overall ≥ baseline−2pp** → strong success. Run full eval to confirm. If full eval ≥ baseline, you're done. Otherwise try Run 2A (compressor).
* Quick-eval **overall < Run 0** → mean_pool *hurt*. Either ditch compression (use Run 0) or try Run 2A/2B with learned compression.
* Quick-eval between Run 0 and baseline → modest improvement, try Run 2 to push further.

**Run 2A (compressor) / Run 2B (two_stream):**
* Quick-eval ≥ Run 1 → progress. Full-eval the better of {Run 1, Run 2}.
* Quick-eval < Run 1 → Run 1 was the winner. Full-eval Run 1.

**Variance reminder:** quick-eval is 1 ep/task — ±10pp per task is normal. Don't chase 2pp deltas in quick eval. Re-run with a different seed if a result is borderline.